In [1]:
%load_ext autoreload
%autoreload 2

from dotenv import load_dotenv
from IPython.display import Markdown
from anthropic import Anthropic
from src.utils.chat import (
    add_user_message,
    add_assistant_message,
    chat_extended,
    combine_text_blocks,
)

load_dotenv()


True

In [2]:
client = Anthropic()
model = "claude-sonnet-4-6"

In [5]:
import json


def generate_dataset():
    prompt = """
    Generate a evaluation dataset for a prompt evaluation. 
    
    The dataset will be used to evaluate prompts that generate Python, JSON, or 
    Regex specifically for AWS-related tasks. Generate an array of JSON objects, 
    each representing a task that requires Python, JSON, or Regex to complete.

    Example output:
    ```json
    [
        {
            "task": "Description of task",
        },
        ...additional
    ]
    ```

    * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
    * Focus on tasks that do not require writing much code
    * Do no tinclude explanations before or after the JSON array.
    * keep the required code short
    * Make every task AWS-related

    Please generate 3 objects.
    """

    messages = []
    add_user_message(messages, prompt)

    text_blocks = chat_extended(client, model=model, messages=messages, temperature=0)

    text = combine_text_blocks(text_blocks)
    try:
        dataset = json.loads(text)
    except json.JSONDecodeError as exc:
        raise ValueError(
            f"Claude did not return valid JSON.\n\n"
            f"Raw text blocks:\n{text_blocks}\n\n"
            f"Combined response:\n{text}\n\n"
        ) from exc

    if len(dataset) != 3:
        raise ValueError(
            f"Expected exactly 3 dataset entries, received {len(dataset)}."
        )

    for index, item in enumerate(dataset):

        if not isinstance(item, dict):

            raise ValueError(

                f"Dataset item {index} must be a JSON object."

            )

        if not isinstance(item.get("task"), str):

            raise ValueError(

                f"Dataset item {index} must contain a string 'task' field."

            )

    return dataset


In [6]:
dataset = generate_dataset()
dataset

[{'task': 'Write a Python function that takes an S3 bucket name and a prefix string, and returns a list of all object keys in that bucket matching the prefix using boto3.'},
 {'task': "Write a JSON object representing an IAM policy that allows read-only access (GetObject and ListBucket) to a specific S3 bucket named 'my-company-data'."},
 {'task': "Write a regular expression that matches a valid Amazon Resource Name (ARN) format, such as 'arn:aws:s3:::my-bucket' or 'arn:aws:iam::123456789012:role/MyRole'."}]